# Text Mining Project — Final Submission Pipeline (`tm_final`)

**Group 12** | Market-sentiment classification: Bearish (0) / Bullish (1) / Neutral (2)

This notebook contains the single, ready-to-run final solution. **Run All** (from a clean state) to reproduce `pred_12.csv`.

**Final model:** concat FinBERT ⊕ twitter-roberta CLS embeddings → MLP(256,128)  
**Validation macro-F1:** 0.808  

> **Note on preprocessing:** the final pipeline feeds **raw text** directly to both
> transformer encoders. This is intentional — each encoder brings its own WordPiece/BPE
> tokenizer that handles casing, sub-words, and out-of-vocabulary tokens. Applying our
> regex-based `clean()` first would discard cashtags and financial tokens the models
> were trained to understand.


## 1. Imports, seeds, and config

In [ ]:
import os, re, json, random, warnings
import numpy as np
import pandas as pd
import torch
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

DATA_DIR  = './data/'
CACHE_DIR = 'cache'
os.makedirs(CACHE_DIR, exist_ok=True)

GROUP_ID     = '12'
LABELS       = [0, 1, 2]
LABEL_NAMES  = {0: 'Bearish', 1: 'Bullish', 2: 'Neutral'}
TARGET_NAMES = [LABEL_NAMES[i] for i in LABELS]
print('Group:', GROUP_ID)


## 2. Load data

In [ ]:
def read_csv_safe(path):
    for enc in ['utf-8', 'latin-1']:
        try: return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError: continue
    raise ValueError(f'Cannot decode {path}')

train_df = read_csv_safe(DATA_DIR + 'train.csv')
test_df  = read_csv_safe(DATA_DIR + 'test.csv')

# Normalise column names
for df in [train_df, test_df]:
    df.columns = [c.strip().lower() for c in df.columns]
    if 'sentence' in df.columns and 'text' not in df.columns:
        df.rename(columns={'sentence': 'text'}, inplace=True)

# Resolve id column
ID_COL = None
for cand in ['id', 'tweet_id', 'index']:
    if cand in test_df.columns:
        ID_COL = cand; break

print(f'Train: {len(train_df)} rows | Test: {len(test_df)} rows')
print(f'Label distribution (train):\n{train_df["label"].value_counts().sort_index()}')


## 3. Embedding extractor (cached)

In [ ]:
from transformers import AutoTokenizer, AutoModel

ENCODERS = {
    'finbert':         'ProsusAI/finbert',
    'twitter_roberta': 'cardiffnlp/twitter-roberta-base-sentiment-latest',
}

def cache_path(name):
    return os.path.join(CACHE_DIR, name)

def load_or_build(name, build_fn):
    path = cache_path(name)
    if os.path.exists(path):
        print(f'[cache] load  {name}')
        return np.load(path)
    print(f'[cache] build {name} ...')
    arr = build_fn()
    np.save(path, arr)
    return arr

@torch.no_grad()
def extract_embeddings(texts, model_name, pooling='cls', batch_size=16, max_length=128):
    tok = AutoTokenizer.from_pretrained(model_name)
    mdl = AutoModel.from_pretrained(model_name, use_safetensors=True); mdl.eval()
    vecs = []
    for i in range(0, len(texts), batch_size):
        batch = [str(x) for x in texts[i:i + batch_size]]
        enc = tok(batch, padding=True, truncation=True,
                  max_length=max_length, return_tensors='pt')
        hs = mdl(**enc).last_hidden_state
        if pooling == 'cls':
            v = hs[:, 0, :]
        else:
            m = enc['attention_mask'].unsqueeze(-1).float()
            v = (hs * m).sum(1) / m.sum(1).clamp(min=1e-9)
        vecs.append(v.cpu().numpy())
        print(f'  {model_name.split("/")[-1]} embedding: {min(i+batch_size, len(texts))}/{len(texts)}', end='\r')
    print()
    return np.vstack(vecs).astype('float32')


## 4. Build full-train and test embeddings

Both encoders receive **raw text** (no regex cleaning). Cached `.npy` files from `tm_tests_12.ipynb` are reused automatically if present.


In [ ]:
def emb_fulltest(enc, pool='cls'):
    m = ENCODERS[enc]
    f = load_or_build(f'emb_{enc}_{pool}_FULLtrain.npy',
                      lambda: extract_embeddings(list(train_df['text']), m, pool))
    t = load_or_build(f'emb_{enc}_{pool}_TEST.npy',
                      lambda: extract_embeddings(list(test_df['text']),  m, pool))
    return f, t

print('Building / loading FinBERT embeddings...')
fb_f, fb_t = emb_fulltest('finbert', 'cls')
print(f'  FinBERT  full-train {fb_f.shape}, test {fb_t.shape}')

print('Building / loading twitter-roberta embeddings...')
tr_f, tr_t = emb_fulltest('twitter_roberta', 'cls')
print(f'  twitter-roberta full-train {tr_f.shape}, test {tr_t.shape}')

# Concatenate the two frozen CLS embeddings -> single feature vector
Xtr_c = np.hstack([fb_f, tr_f])
Xte_c = np.hstack([fb_t, tr_t])
print(f'Concat feature: train {Xtr_c.shape}, test {Xte_c.shape}')


## 5. Fit MLP on full training set and predict test

Same `MLPClassifier(256, 128)` architecture and `random_state=42` as in `tm_tests_12.ipynb`.


In [ ]:
from sklearn.neural_network import MLPClassifier

final_clf = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    max_iter=400,
    early_stopping=True,
    random_state=RANDOM_STATE,
).fit(Xtr_c, train_df['label'])

test_pred = final_clf.predict(Xte_c)
print('Predicted distribution:', dict(pd.Series(test_pred).value_counts().sort_index()))


## 6. Write `pred_12.csv` and validate

In [ ]:
ids = test_df[ID_COL].values if ID_COL else np.arange(len(test_df))
pred_out = pd.DataFrame({'id': ids, 'label': np.asarray(test_pred).astype(int)})

assert len(pred_out) == len(test_df), f'Row count mismatch: {len(pred_out)} vs {len(test_df)}'
assert set(pred_out['label'].unique()).issubset({0, 1, 2}), 'Labels must be 0/1/2'

out_name = f'pred_{GROUP_ID}.csv'
pred_out.to_csv(out_name, index=False)
print(f'Wrote {out_name} with {len(pred_out)} rows')
print('Sample:')
pred_out.head()


### Done

`pred_12.csv` should be byte-identical to the one produced by `tm_tests_12.ipynb` (same seeds, same model, same cached embeddings).
